# 05 — Plant vs modèle identifié (anti-circularité)

Même entrée u, deux sorties : le **plant** (solaire aussi sur la masse)
et un R2C2 **sans** alpha_s,mass (modèle interne).
L'écart est volontaire : on ne pilote pas le RC appris sur lui-même.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from basic_mpc.control.internal import r2c2_internal_from_plant
from basic_mpc.models.plant import ThermalPlant, literature_plant_params, synthetic_weather
from basic_mpc.models.r2c2 import discretize as discretize_r2

plant_p = literature_plant_params()
n = int(48 * 3600 / plant_p.dt_seconds)
w = synthetic_weather(n, plant_p.dt_seconds, seed=4)
t_ext = w["t_ext"].to_numpy()
solar = w["S"].to_numpy()
heating = np.clip(20.0 - t_ext, 0, None) * 8.0
plant = ThermalPlant(params=plant_p, x0=np.array([18.0, 18.0]), seed=0)
traj = plant.simulate(t_ext, solar, heating)

internal = r2c2_internal_from_plant(plant_p)
ad, bd = discretize_r2(internal)
x = np.array([18.0, 18.0])
ta_mod = np.empty(n)
for k in range(n):
    x = ad @ x + bd @ np.array([t_ext[k], solar[k], heating[k]])
    ta_mod[k] = x[0]
hours = np.arange(n) * plant_p.dt_seconds / 3600

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(hours, traj["ta_true"], color="#2c2416", label="plant (α_s,mass)")
ax.plot(hours, ta_mod, color="#3d6b6b", label="R2C2 interne (pas de α_s,mass)")
ax.set_xlabel("heures")
ax.set_ylabel("T_air (°C)")
ax.set_title("Même u : l'écart est la misspecification, pas un bug")
ax.legend(frameon=False)
plt.show()
print("RMSE air plant vs interne",
      float(np.sqrt(np.mean((traj["ta_true"] - ta_mod) ** 2))))